# MoveBench: Global Trajectory Map + Coverage Ideas

This notebook:

1. Loads **all studies** under `MOVEBENCH_DATA_ROOT/studies_10ind/` (i.e., the full benchmark dataset root).
2. Builds per-individual trajectory polylines.
3. Saves **static** global plots (PNG) of trajectories and point density.
4. Brainstorms additional ways to visualize dataset **coverage** (space/time/environment).

If you need plotting deps:

```bash
pip install -e ".[apps]"
```

Notes:
- By default, images are written under `pngs/` at the repo root.
- If `cartopy` is installed, the plots include coastlines; otherwise they fall back to a plain lon/lat canvas.
- For very large benchmarks, increase downsampling to keep rendering fast.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# --- Repo + data root wiring ---
# If you're not running from repo root, uncomment and set this:
# os.chdir("/path/to/movebench")

# Find repo root by walking upward until we see the package directory.
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'movebench').exists():
    repo_root = repo_root.parent
if not (repo_root / 'movebench').exists():
    raise RuntimeError('Could not find repo root (expected a movebench/ directory). Set os.chdir(...) first.')

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# If MOVEBENCH_DATA_ROOT isn't configured, default to the repo-local data path
# so Study.list_available() works out-of-the-box for this checkout.
_default_data_root = (repo_root / 'data' / 'movebench').resolve()
if 'MOVEBENCH_DATA_ROOT' not in os.environ and _default_data_root.exists():
    os.environ['MOVEBENCH_DATA_ROOT'] = str(_default_data_root)

print('repo_root:', repo_root)
print('MOVEBENCH_DATA_ROOT:', os.environ.get('MOVEBENCH_DATA_ROOT'))


In [ ]:
import colorsys
import hashlib

import numpy as np
import pandas as pd

from movebench import Study
from movebench.trajectories import INDIVIDUAL_COL, LAT_COL, LON_COL, OUTLIER_COL, SET_COL, TIME_COL

try:
    import matplotlib.pyplot as plt
    from matplotlib.collections import LineCollection
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        'matplotlib is required for the static plots. Install with: pip install -e ".[apps]" (from repo root).'
    ) from e

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

    HAS_CARTOPY = True
except ModuleNotFoundError:
    HAS_CARTOPY = False


In [ ]:
# --- Discover studies (full benchmark = everything under studies root) ---
study_ids = Study.list_available()
print("n_studies:", len(study_ids))
study_ids[:20]


In [ ]:
# --- Load trajectories across studies ---
# Keep it lightweight: only load columns we need for mapping.
USECOLS = [INDIVIDUAL_COL, LAT_COL, LON_COL, TIME_COL, SET_COL, OUTLIER_COL]


def _load_one_study(study_id: str) -> pd.DataFrame:
    p = Study(study_id).trajectories_path

    def _use(c: str) -> bool:
        return c in USECOLS

    df = pd.read_csv(p, usecols=_use)
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.dropna(subset=[TIME_COL, LAT_COL, LON_COL])

    if OUTLIER_COL in df.columns:
        # Treat any truthy value as an outlier marker.
        df = df.loc[~df[OUTLIER_COL].fillna(False).astype(bool)].copy()

    df["study_id"] = str(study_id)
    return df


dfs = []
for sid in study_ids:
    dfs.append(_load_one_study(sid))

traj_df = pd.concat(dfs, ignore_index=True)

print("rows:", len(traj_df))
print("studies:", traj_df["study_id"].nunique())
print("individuals:", traj_df[INDIVIDUAL_COL].nunique())
print("sets:", sorted(traj_df[SET_COL].dropna().unique().tolist()))
traj_df.head()


In [ ]:
# --- Build polylines (one per (study, individual, set)) ---
MAX_POINTS_PER_PATH = 2_000
SPLIT_AT_ANTIMERIDIAN = True


def _uniform_downsample(points: list[list[float]], max_points: int) -> list[list[float]]:
    n = len(points)
    if n <= max_points:
        return points
    idx = np.linspace(0, n - 1, num=max_points, dtype=np.int64)
    return [points[int(i)] for i in idx]


def _split_antimeridian(points: list[list[float]]) -> list[list[list[float]]]:
    """Split a path when it jumps across the +/-180° seam."""
    if len(points) <= 1:
        return [points]
    out: list[list[list[float]]] = []
    cur: list[list[float]] = [points[0]]
    for lon, lat in points[1:]:
        prev_lon = cur[-1][0]
        if np.isfinite(prev_lon) and np.isfinite(lon) and abs(float(lon) - float(prev_lon)) > 180.0:
            if len(cur) >= 2:
                out.append(cur)
            cur = [[lon, lat]]
        else:
            cur.append([lon, lat])
    if len(cur) >= 2:
        out.append(cur)
    return out if out else [points]


def _stable_rgb(key: str) -> tuple[float, float, float]:
    # Stable study color assignment across runs (for COLOR_BY='study').
    h = int(hashlib.md5(key.encode('utf-8')).hexdigest()[:8], 16) / float(0xFFFFFFFF)
    r, g, b = colorsys.hsv_to_rgb(h, 0.75, 0.9)
    return (float(r), float(g), float(b))


path_rows: list[dict] = []

group_cols = ['study_id', INDIVIDUAL_COL, SET_COL]
for (study_id, indiv, split), g in traj_df.sort_values(TIME_COL).groupby(group_cols, sort=False):
    pts = g[[LON_COL, LAT_COL]].to_numpy(dtype=np.float64, copy=False)
    pts = pts[np.isfinite(pts).all(axis=1)]
    if len(pts) < 2:
        continue

    path = pts.tolist()  # [[lon, lat], ...]
    path = _uniform_downsample(path, MAX_POINTS_PER_PATH)

    segments = _split_antimeridian(path) if SPLIT_AT_ANTIMERIDIAN else [path]
    for seg in segments:
        if len(seg) < 2:
            continue
        path_rows.append(
            {
                'seg': np.asarray(seg, dtype=np.float32),
                'study_id': str(study_id),
                'individual': str(indiv),
                'split': str(split),
                'n': int(len(seg)),
            }
        )

print('n_paths (segments):', len(path_rows))
print('train segments:', sum(1 for r in path_rows if r['split'] == 'train'))
print('test segments:', sum(1 for r in path_rows if r['split'] == 'test'))


In [ ]:
# --- Static global trajectory plot (saved as PNG) ---
OUT_DIR = (repo_root / 'pngs').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

COLOR_BY = 'split'  # 'split' or 'study'

train_color = (0.12, 0.47, 0.71, 0.15)  # RGBA
test_color = (1.00, 0.50, 0.05, 0.22)

segs_train = [r['seg'] for r in path_rows if r['split'] == 'train']
segs_test = [r['seg'] for r in path_rows if r['split'] == 'test']

if HAS_CARTOPY:
    proj = ccrs.Robinson()
    fig = plt.figure(figsize=(16, 8))
    ax = plt.axes(projection=proj)
    ax.set_global()
    ax.add_feature(cfeature.LAND.with_scale('110m'), facecolor='0.95', zorder=0)
    ax.add_feature(cfeature.OCEAN.with_scale('110m'), facecolor='white', zorder=0)
    ax.coastlines(resolution='110m', linewidth=0.6, color='0.35', zorder=1)

    if COLOR_BY == 'study':
        colors = [(*_stable_rgb(r['study_id']), 0.18 if r['split'] == 'train' else 0.28) for r in path_rows]
        lc = LineCollection([r['seg'] for r in path_rows], colors=colors, linewidths=0.45, transform=ccrs.PlateCarree())
        ax.add_collection(lc)
    else:
        if segs_train:
            lc_tr = LineCollection(segs_train, colors=[train_color], linewidths=0.45, transform=ccrs.PlateCarree())
            ax.add_collection(lc_tr)
        if segs_test:
            lc_te = LineCollection(segs_test, colors=[test_color], linewidths=0.55, transform=ccrs.PlateCarree())
            ax.add_collection(lc_te)

else:
    fig, ax = plt.subplots(figsize=(16, 8))
    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, linewidth=0.3, color='0.85')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

    if COLOR_BY == 'study':
        colors = [(*_stable_rgb(r['study_id']), 0.18 if r['split'] == 'train' else 0.28) for r in path_rows]
        lc = LineCollection([r['seg'] for r in path_rows], colors=colors, linewidths=0.45)
        ax.add_collection(lc)
    else:
        if segs_train:
            lc_tr = LineCollection(segs_train, colors=[train_color], linewidths=0.45)
            ax.add_collection(lc_tr)
        if segs_test:
            lc_te = LineCollection(segs_test, colors=[test_color], linewidths=0.55)
            ax.add_collection(lc_te)

ax.set_title('MoveBench trajectories (global)')
out_path = OUT_DIR / 'global_trajectories.png'
fig.savefig(out_path, dpi=220, bbox_inches='tight')
print('wrote:', out_path)
plt.show()


In [ ]:
# --- Point density heatmap (saved as PNG) ---
OUT_DIR = (repo_root / 'pngs').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

lons = traj_df[LON_COL].to_numpy(dtype=np.float64, copy=False)
lats = traj_df[LAT_COL].to_numpy(dtype=np.float64, copy=False)
m = np.isfinite(lons) & np.isfinite(lats)
lons = lons[m]
lats = lats[m]

# 1-degree bins (tweak for resolution vs speed)
lon_edges = np.linspace(-180, 180, 361)
lat_edges = np.linspace(-90, 90, 181)
H, yedges, xedges = np.histogram2d(lats, lons, bins=[lat_edges, lon_edges])
Hlog = np.log10(H + 1.0)

extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]

if HAS_CARTOPY:
    fig = plt.figure(figsize=(16, 8))
    ax = plt.axes(projection=ccrs.Robinson())
    ax.set_global()
    ax.add_feature(cfeature.LAND.with_scale('110m'), facecolor='0.95', zorder=0)
    ax.add_feature(cfeature.OCEAN.with_scale('110m'), facecolor='white', zorder=0)
    ax.coastlines(resolution='110m', linewidth=0.6, color='0.35', zorder=2)
    im = ax.imshow(Hlog, extent=extent, origin='lower', transform=ccrs.PlateCarree(), cmap='magma', zorder=1)
else:
    fig, ax = plt.subplots(figsize=(16, 8))
    im = ax.imshow(Hlog, extent=extent, origin='lower', cmap='magma', aspect='auto')
    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

ax.set_title('MoveBench point density (log10(count + 1))')
cb = plt.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
cb.set_label('log10(count + 1)')

out_path = OUT_DIR / 'global_point_density_1deg.png'
fig.savefig(out_path, dpi=220, bbox_inches='tight')
print('wrote:', out_path)
plt.show()


In [ ]:
# --- Quick per-study coverage table (space/time) ---
summary = (
    traj_df.groupby("study_id")
    .agg(
        n_points=(LAT_COL, "size"),
        n_individuals=(INDIVIDUAL_COL, "nunique"),
        t_min=(TIME_COL, "min"),
        t_max=(TIME_COL, "max"),
        lat_min=(LAT_COL, "min"),
        lat_max=(LAT_COL, "max"),
        lon_min=(LON_COL, "min"),
        lon_max=(LON_COL, "max"),
    )
    .sort_values(["n_points"], ascending=False)
)

summary


## Coverage visualization brainstorm

Here are additional ways to visualize **coverage** of the benchmark beyond the global trajectory map.

### Spatial coverage
- **Grid occupancy**: count points per lat/lon grid cell (e.g., 1°, 2°, 5°) and plot a world heatmap.
- **Study footprints**: bounding boxes / convex hulls per study; overlap map.
- **Per-biome/eco-region**: intersect points with biome polygons (e.g., WWF ecoregions) and show counts.
- **Coastal vs inland**: histogram of distance-to-coast / distance-to-water for observed points.

### Temporal coverage
- **Seasonality**: month-of-year heatmaps per study/individual; compare train vs test.
- **Local time-of-day**: (if timestamps in local time or convertible via timezone inference) diurnal coverage.
- **Sampling cadence**: distribution of time gaps between fixes; missingness patterns.

### Movement-behavior coverage (from trajectories only)
- **Step length vs turn angle**: 2D density (behavioral space) per study and globally.
- **Burst structure**: distribution of burst lengths (continuous segments) and gap-driven breaks.

### Environmental coverage (from rasters / point feature stores)
- **Land cover classes**: histograms of MODIS LC (`modis_lc`) at observed points; compare per study & split.
- **Vegetation**: NDVI/EVI distributions; seasonal NDVI curves; tree cover fraction (MODIS CVF).
- **Climate space**: scatter/PCA/UMAP of `bioclim` variables for observed points; compare train vs test.
- **Human influence**: histogram/quantiles of Human Impact Index (`hii`).
- **Dynamic World**: per-class probability histograms; argmax class map of visited locations.

### Dataset balance & bias
- **Geographic bias**: points per continent/country (requires polygon overlay), latitudinal bands.
- **Study contribution**: Pareto chart of points/individuals per study.
- **Train/test shift**: compare spatial density and covariate distributions between splits.


## (Optional) Coverage from point feature stores (env distributions)

If you have extracted point feature stores (SSF/iSSF), you can use them to compute land cover / vegetation / climate distributions at the **true** next-step endpoints.

This avoids re-sampling rasters yourself, and uses the same covariates the models see.

Important gotchas:
- Categorical layers (e.g. `modis_lc`) are stored as **one-hot channels** with band ids like `modis_lc.class_7` (not `modis_lc.LC_Type1`).
- If your stores were extracted with `configs/layers_local.yaml`, you will not have `modis_lc`; use `dynamic_world.*` bands instead for land-cover-ish coverage.


In [ ]:
# NOTE: This section requires torch + extracted point stores.
# It is written to fail gracefully if those aren't available yet.

try:
    import torch

    from movebench.features.manifest import read_manifest
    from movebench.features.points import select_point_store_dir
except ModuleNotFoundError as e:
    print("skip env-coverage section (missing dep):", e)
    torch = None


In [ ]:
if torch is not None:
    from datetime import datetime

    # Match benchmark defaults (configs/benchmarks/all.yaml)
    SAMPLER = {
        'expected_type': 'parametric',
        'num_candidates': 100,
        'random_seed': 0,
        'step_multiplier': 2.0,
        'step_q': 0.95,
        'gap_q': 0.999,
        'bounds_buffer_deg': 0.01,
    }

    # MODIS LC (IGBP) class names: 0..17
    MODIS_LC_IGBP = {
        0: 'Water',
        1: 'Evergreen Needleleaf Forest',
        2: 'Evergreen Broadleaf Forest',
        3: 'Deciduous Needleleaf Forest',
        4: 'Deciduous Broadleaf Forest',
        5: 'Mixed Forests',
        6: 'Closed Shrublands',
        7: 'Open Shrublands',
        8: 'Woody Savannas',
        9: 'Savannas',
        10: 'Grasslands',
        11: 'Permanent Wetlands',
        12: 'Croplands',
        13: 'Urban and Built-Up',
        14: 'Cropland/Natural Vegetation Mosaic',
        15: 'Snow and Ice',
        16: 'Barren or Sparsely Vegetated',
        17: 'Unclassified',
    }

    DYNAMIC_WORLD_CLASSES = [
        'water',
        'trees',
        'grass',
        'flooded_vegetation',
        'crops',
        'shrub_and_scrub',
        'built',
        'bare',
        'snow_and_ice',
    ]


    def _parse_created_at(s: str):
        try:
            return datetime.fromisoformat(str(s).replace('Z', '+00:00'))
        except Exception:
            return datetime.min


    def find_point_store_dir(study: Study, *, split: str):
        """Find a point store dir for a study/split matching SAMPLER (most recent if multiple)."""
        base = study.study_dir / 'features' / 'points' / str(split)
        if not base.exists():
            return None, None

        matches = []
        for config_dir in sorted(base.iterdir()):
            if not config_dir.is_dir():
                continue
            for sampler_dir in sorted(config_dir.iterdir()):
                if not sampler_dir.is_dir():
                    continue
                mp = sampler_dir / 'manifest.json'
                fp = sampler_dir / 'features.pt'
                if not mp.exists() or not fp.exists():
                    continue
                try:
                    m = read_manifest(mp)
                except Exception:
                    continue
                s = m.sampler
                e = m.extraction
                if str(s.get('type')) != str(SAMPLER['expected_type']):
                    continue
                if int(s.get('num_candidates')) != int(SAMPLER['num_candidates']):
                    continue
                if int(s.get('random_seed')) != int(SAMPLER['random_seed']):
                    continue
                if float(s.get('step_multiplier')) != float(SAMPLER['step_multiplier']):
                    continue
                if float(s.get('step_q')) != float(SAMPLER['step_q']):
                    continue
                if float(s.get('gap_q')) != float(SAMPLER['gap_q']):
                    continue
                if float(e.get('bounds_buffer_deg')) != float(SAMPLER['bounds_buffer_deg']):
                    continue

                matches.append((sampler_dir, m))

        if not matches:
            return None, None

        matches.sort(key=lambda x: _parse_created_at(x[1].created_at))
        return matches[-1]


    def load_true_env_for_study(study_id: str, *, split: str):
        """Return (band_ids_in_order, env_true) where env_true is (N, C) at the true endpoint."""
        study = Study(study_id)
        store_dir, manifest = find_point_store_dir(study, split=split)
        if store_dir is None:
            raise FileNotFoundError('No matching point store found')

        band_ids = manifest.layer_plan.band_ids_in_order()

        tensors = torch.load(store_dir / 'features.pt', map_location='cpu')
        env = tensors['env']  # (N, C, P)
        valid = tensors['valid']  # (N, C, P)

        true_idx = int(env.shape[2] - 1)  # last index is the true endpoint
        env_true = env[:, :, true_idx]
        valid_true = valid[:, :, true_idx].to(torch.bool)

        env_true = torch.where(valid_true, env_true, torch.full_like(env_true, float('nan')))
        return band_ids, env_true.detach().cpu().numpy().astype(np.float32, copy=False)


    def land_cover_histogram(*, split: str = 'train') -> pd.DataFrame:
        counts: dict[str, int] = {}
        scheme_used: dict[str, str] = {}

        for sid in study_ids:
            try:
                band_ids, env_true = load_true_env_for_study(sid, split=split)
            except Exception as e:
                print(f'[{sid}] skip (no store or load failed): {type(e).__name__}: {e}')
                continue

            # Prefer MODIS LC if present; else fall back to Dynamic World.
            modis_idxs = [i for i, b in enumerate(band_ids) if b.startswith('modis_lc.class_')]
            if modis_idxs:
                X = env_true[:, modis_idxs]
                m = np.isfinite(X).any(axis=1)
                if m.any():
                    # One-hot; argmax recovers class id.
                    cls = np.argmax(np.nan_to_num(X[m], nan=-np.inf), axis=1).astype(int)
                    for k, v in zip(*np.unique(cls, return_counts=True)):
                        name = MODIS_LC_IGBP.get(int(k), f'class_{int(k)}')
                        counts[name] = counts.get(name, 0) + int(v)
                    scheme_used[sid] = 'modis_lc'
                continue

            dw_bands = [f'dynamic_world.{c}' for c in DYNAMIC_WORLD_CLASSES]
            if all(b in band_ids for b in dw_bands):
                idxs = [band_ids.index(b) for b in dw_bands]
                X = env_true[:, idxs]
                m = np.isfinite(X).any(axis=1)
                if m.any():
                    cls = np.argmax(np.nan_to_num(X[m], nan=-np.inf), axis=1).astype(int)
                    for k, v in zip(*np.unique(cls, return_counts=True)):
                        name = DYNAMIC_WORLD_CLASSES[int(k)]
                        counts[name] = counts.get(name, 0) + int(v)
                    scheme_used[sid] = 'dynamic_world'
                continue

            print(f'[{sid}] skip (no land-cover-like bands found)')

        out = (
            pd.DataFrame({'class': list(counts.keys()), 'count': list(counts.values())})
            .sort_values('count', ascending=False)
            .reset_index(drop=True)
        )
        print('schemes used:', pd.Series(list(scheme_used.values())).value_counts().to_dict())
        return out


    lc_hist = land_cover_histogram(split='train')
    lc_hist
